In [ ]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Downloading the ImageNet10 dataset from kaggle

In [ ]:
!curl -L -o /content/imagenet10.zip https://www.kaggle.com/api/v1/datasets/download/liusha249/imagenet10
print("Extracting dataset (quiet unzip)...")
!unzip -q /content/imagenet10.zip -d /content/yolov1-cupy
print("Done.")
!rm /content/imagenet10.zip

## Pretraining (Mini Darknet, ImageNet10)

A few epochs of SGD over the full dataset. Each epoch uses a new shuffle (`seed = epoch`). Re-run after dataset + path cells.

| Block | Layers                                                           | Output size   |
|-------|------------------------------------------------------------------|--------------|
| Input | —                                                                | 3 × 224 × 224|
| 1     | Conv2D(3→16, 3, pad=1) → BN → LeakyReLU → MaxPool(2)            | 16 × 112 × 112|
| 2     | Conv2D(16→32, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 32 × 56 × 56 |
| 3     | Conv2D(32→64, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 64 × 28 × 28 |
| 4     | Conv2D(64→128, 3, pad=1) → BN → LeakyReLU → MaxPool(2)          | 128 × 14 × 14|
| 5     | Conv2D(128→256, 3, pad=1) → BN → LeakyReLU → MaxPool(2)         | 256 × 7 × 7  |
| Head  | AvgPool(7) → Flatten → Linear(256, 10)                          | 10           |

In [ ]:
from pathlib import Path

import cupy as cp
from tqdm.auto import tqdm

from cross_entropy import softmax_cross_entropy_grad, softmax_cross_entropy_loss
from image_batch_loader import dataset_size, image_label_batch, num_batches_per_epoch
from mini_darknet import MiniDarknet

REPO = "/content/yolov1-cupy"
batch_size = 8
learning_rate = 0.01
num_epochs = 2

n_samples = dataset_size(REPO)
n_batches = num_batches_per_epoch(REPO, batch_size)
print(f"samples: {n_samples}, batches per epoch: {n_batches}")

model = MiniDarknet(num_classes=10)

for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    pbar = tqdm(
        range(n_batches),
        desc=f"Epoch {epoch + 1}/{num_epochs}",
        unit="batch",
        mininterval=0.5,
    )
    for batch_idx in pbar:
        x, y_cpu = image_label_batch(
            REPO,
            batch_size=batch_size,
            seed=epoch,
            batch_index=batch_idx,
        )
        model.zero_grad()
        logits = model.forward(x)
        y_gpu = cp.asarray(y_cpu, dtype=cp.int64)
        pred = cp.argmax(logits, axis=1)
        epoch_correct += int(cp.sum(pred == y_gpu))
        epoch_total += int(y_gpu.shape[0])

        batch_loss = softmax_cross_entropy_loss(
            logits, y_cpu, mean_over_batch=True
        )
        epoch_loss += batch_loss
        grad_logits = softmax_cross_entropy_grad(
            logits, y_cpu, mean_over_batch=True
        )
        model.backward(grad_logits)
        model.sgd_step(learning_rate)
        running_mean = epoch_loss / (batch_idx + 1)
        running_acc = epoch_correct / epoch_total
        pbar.set_postfix_str(f"loss={running_mean:.4f} acc={running_acc:.3f}")
    mean_loss = epoch_loss / n_batches
    train_acc = epoch_correct / epoch_total
    print(
        f"epoch {epoch + 1}/{num_epochs} done — "c
        f"mean loss {mean_loss:.4f}  train acc {train_acc:.4f} ({epoch_correct}/{epoch_total})"
    )

weights_path = Path(REPO) / "mini_darknet_pretrained.npz"
model.save_weights(weights_path)
print(f"saved weights to {weights_path.resolve()}")